In [ ]:
import cv2, os, re, json, tempfile, hashlib
from retinaface import RetinaFace
import libreface

video_path = "/Users/erynw/(Research)/Libreface/Videos/group-5-task.mp4"
output_dir = "/Users/erynw/(Research)/Libreface/libreface-labels"
os.makedirs(output_dir, exist_ok=True)

base = os.path.splitext(os.path.basename(video_path))[0]
m = re.search(r"group-(\d+)", base)
group_id = int(m.group(1)) if m else None

output_json_path = os.path.join(output_dir, f"{base}_libreface.json")
output_timelapse_path = os.path.join(output_dir, f"{base}_timelapse.mp4")

process_fps = 1    # analyze 1 frame per second
playback_fps = 15  # playback speed for timelapse

# Helper functions
def clip_box(x1, y1, x2, y2, w, h):
    x1 = max(0, min(int(x1), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    x2 = max(0, min(int(x2), w))
    y2 = max(0, min(int(y2), h))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    boxBArea = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-5)

def hash_face(crop):
    """Create a quick hash so similar frames reuse cached emotion results."""
    h = hashlib.sha1(crop.tobytes()).hexdigest()
    return h[:16]  # short hash

# Video capture
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
H, W = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

stride = max(int(fps / process_fps), 1)

out_timelapse = cv2.VideoWriter(
    output_timelapse_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    playback_fps,
    (W, H)
)

results, prev_faces = [], []
emotion_cache = {}  # hash -> emotion

# Main loop
for frame_idx in range(0, total_frames, stride):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        continue

    detections = RetinaFace.detect_faces(frame)
    if not detections:
        continue

    # Gets the top 3 faces
    faces = []
    for det in detections.values():
        x1, y1, x2, y2 = det["facial_area"]
        score = det.get("score", 0.0)
        box = clip_box(x1, y1, x2, y2, W, H)
        if box:
            faces.append((box, score))

    if len(faces) < 3:
        continue

    faces = sorted(faces, key=lambda x: x[1], reverse=True)[:3]
    faces = sorted(faces, key=lambda x: x[0][0])

    # Track faces across frames
    current = []
    if prev_faces:
        used_prev = set()
        for box, _ in faces:
            best_iou, best_id = 0, None
            for pid, pbox in prev_faces:
                val = iou(box, pbox)
                if val > best_iou:
                    best_iou, best_id = val, pid
            if best_iou > 0.3 and best_id not in used_prev:
                current.append((best_id, box))
                used_prev.add(best_id)
            else:
                new_id = f"face-{len(current)+1}"
                current.append((new_id, box))
    else:
        current = [(f"face-{i+1}", box) for i, (box, _) in enumerate(faces)]

    prev_faces = current.copy()

    # Emotion Recog.
    emotions = {}
    for fid, (x1, y1, x2, y2) in current:
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            emotions[fid] = "Unknown"
            continue

        face_hash = hash_face(crop)
        if face_hash in emotion_cache:
            emotion = emotion_cache[face_hash]
        else:
            try:
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crop_resized = cv2.resize(crop_rgb, (224, 224))

                with tempfile.NamedTemporaryFile(suffix=".jpg") as tmp:
                    cv2.imwrite(tmp.name, cv2.cvtColor(crop_resized, cv2.COLOR_RGB2BGR))
                    attrs = libreface.get_facial_attributes(tmp.name)
                    emotion = attrs.get("facial_expression", "Unknown")

                emotion_cache[face_hash] = emotion
            except Exception as e:
                print(f"[WARN] Libreface failed on {fid}: {e}")
                emotion = "Unknown"

        emotions[fid] = emotion

        # Draws the annotated boxes
        color = (0, 255, 0) if fid == "face-1" else (255, 255, 0) if fid == "face-2" else (0, 128, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{fid}: {emotion}", (x1, max(25, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Saves frame data and write timelapse frame
    results.append({
        "group_id": group_id,
        "videoTime": int(frame_idx / fps),
        "frame_number": frame_idx,
        "libreface-emotions": emotions
    })
    out_timelapse.write(frame)

cap.release()
out_timelapse.release()


# Save JSON results

with open(output_json_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Timelapse: {output_timelapse_path}")
print(f"JSON: {output_json_path}")
print(f"Cached emotions: {len(emotion_cache)} unique faces processed.")

Using device: cpu for inference...


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu fo

Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-1: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Us

Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
[WARN] Libreface failed on face-1: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu

Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-1: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu 

Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Us

Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
[WARN] Libreface failed on face-3: No face landmarks
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu f

Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu for inference...
Using device: cpu fo